In [0]:
# CELL 1: Install & Verify Libraries
# -------------------------------------------------------------------------
# 1. Install the library
%pip install beautifulsoup4
 
# 2. Verify installation immediately
try:
    from bs4 import BeautifulSoup
    print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:
    print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")


In [0]:
# CELL 2: Setup & Dynamic Link Finder (Network Safe Version)
# -------------------------------------------------------------------------
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, timedelta
 
# 1. Define URL
base_page_url = "https://www.dubaipulse.gov.ae/data/dld-transactions/dld_transactions-open"
 
# 2. Local Cluster Storage Paths
landing_zone = "/tmp/transactions/landing/"
archive_zone = "/tmp/transactions/archive/"
 
os.makedirs(landing_zone, exist_ok=True)
os.makedirs(archive_zone, exist_ok=True)

# 3. Dynamic Link Finder with Network Fallbacks
def get_dynamic_download_url():
    print(f"🔍 Scraping {base_page_url}...")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
 
    try:
        # Added a 15-second connection timeout so it doesn't hang forever
        response = requests.get(base_page_url, headers=headers, timeout=15)
        response.raise_for_status()
 
        soup = BeautifulSoup(response.content, 'html.parser')
 
        for link in soup.find_all('a', href=True):
            href = link['href']
            if "download" in href and "transactions.csv" in href:
                final_url = f"https://www.dubaipulse.gov.ae{href}" if href.startswith("/") else href
                print(f"✅ Found dynamic link: {final_url}")
                return final_url
                
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Network connection to portal timed out or failed: {str(e)}")
        print("💡 Falling back to the known direct download URL structure...")
        
        # Hardcoded Fallback URL to bypass the scraping page if the portal's front-end is down
        fallback_url = "https://www.dubaipulse.gov.ae/data/dld-transactions/download?transactions.csv"
        return fallback_url
 
    raise Exception("❌ Could not find or resolve the download link.")

print("Configuration loaded seamlessly with network safety fallbacks!")

In [0]:
# CELL 3: Retention Policy
# -------------------------------------------------------------------------
def cleanup_old_files(days_to_keep=14):
    print("🧹 Checking archive for old files...")
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)
 
    try:
        files = dbutils.fs.ls(archive_zone)
        for f in files:
            # Expected format: YYYY-MM-DD.csv
            filename = f.name.replace(".csv", "")
            try:
                file_date = datetime.strptime(filename, "%Y-%m-%d")
                if file_date < cutoff_date:
                    print(f" 🗑️ Deleting: {f.name}")
                    dbutils.fs.rm(f.path)
            except ValueError:
                continue
    except Exception:
        print(" ℹ️ Archive is empty or does not exist yet.")
 
# This runs the function and must be completely unindented
cleanup_old_files()

In [0]:
# CELL 4: Archival Logic
# -------------------------------------------------------------------------
import shutil

def archive_current_file():
    current_file_path = f"{landing_zone}current.csv"
 
    if os.path.exists(current_file_path):
        # Rename to YYYY-MM-DD.csv
        today_str = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone}{today_str}.csv"
 
        print(f"📦 Archiving previous 'current.csv' to: {archive_path}")
        shutil.move(current_file_path, archive_path)
    else:
        print("ℹ️ No 'current.csv' found in landing. First run?")
 
archive_current_file()

In [0]:
# CELL 5: Download Snapshot (Robust Version with Retries)
# -------------------------------------------------------------------------
import os
import time
import requests  # Added missing import
 
#def download_snapshot_with_retry(max_retries=3):
    fresh_url = get_dynamic_download_url()
    
    # This directly points to the folder structure we built in Cell 2
    target_path = f"{landing_zone}current.csv"
    headers = {'User-Agent': 'Mozilla/5.0'}
 
    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Attempt {attempt}/{max_retries}: Downloading directly to cluster storage...")
 
            # 30-second timeout, Stream=True
            with requests.get(fresh_url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(target_path, 'wb') as f:
                    # 10MB Chunks
                    for chunk in r.iter_content(chunk_size=10*1024*1024):
                        if chunk:
                            f.write(chunk)
 
            print("✅ Success! 'current.csv' is saved and ready.")
            break  # Exit loop on success
 
        except Exception as e:
            print(f" ⚠️ Attempt {attempt} failed: {str(e)}")
            if attempt < max_retries:
                print(" ⏳ Waiting 5 seconds...")
                time.sleep(5)
            else:
                print("❌ All retries failed.")
                raise e
 
# Run it
download_snapshot_with_retry()